# Allergen Detection from Ingredient Lists
## Multi-label Classification with MobileBERT

This notebook trains a multi-label classifier to detect 8 allergens from product ingredient texts.

**Updated to use utility modules for reduced code duplication and improved maintainability**

## Overview

Trains MobileBERT model for multi-label allergen classification.

## Workflow

1. Load and prepare data using utility modules
2. Split into train/val/test with stratified splitting
3. Train model with weighted loss
4. Evaluate and compute optimal thresholds
5. Save final model

## 1. Setup & Imports

Using modular utility functions to reduce code duplication

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import ast
import shutil
import json
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score
from scipy.special import expit
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Import project utility modules
from utils.data_utils import (
    get_data_directories,
    load_labeled_data,
    create_stratified_splits,
    augment_dataframe
)
from utils.text_processing import (
    clean_html,
    parse_label_string,
    get_allergen_list,
    allergens_to_binary
)
from utils.model_utils import (
    compute_class_weights,
    find_best_thresholds,
    apply_thresholds
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis
)

# Set seeds for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("All imports completed using utility modules.")

All imports completed using utility modules.


## 2. Load and Prepare Data

In [2]:
# Get project directories for consistent path handling
dirs = get_data_directories()
print(f"Project base directory: {dirs['base']}")

# Load labeled data using utility function
# Use the enhanced dataset which contains detected allergens
df = load_labeled_data(filepath=os.path.join(dirs['final'], 'labeled_dataset_enhanced.csv'))
print(f"Dataset shape: {df.shape}")
df.head()

Project base directory: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION
Dataset shape: (1057, 11)


,code,brands,product_name_en,ingredients_text_en,detected_allergens,official_allergens_mapped,traces_allergens,consensus_allergens,combined_allergens,may_contain,coconut
0,8850389100684,Sappe Public Company Limited,Mogu litchi,"water, lychee juice from concentrate 25%, coco...",[],[],[],[],"{'detected_only': [], 'detected_or_official': ...",[],['coconut']
1,4800194105958,Oishi,Oishi Prawn Crackers (Sweet Extra Flavor),"wheat, tapioca starch, vegetable oil (may cons...",['wheat'],[],[],[],"{'detected_only': ['wheat'], 'detected_or_offi...",[],['coconut']
2,4806533726822,Gardenia bakeries (Phils) Inc,Neu Bake,"high protein wheat flour (with vitamins b1, b2...","['wheat', 'milk']",[],[],[],"{'detected_only': ['milk', 'wheat'], 'detected...",[],['coconut']
3,4800361316934,Maggi,Savor Liquid Seasoning Chilimansi,"water, lodized salt, sugar, flavor enhancers (...",['wheat'],[],[],[],"{'detected_only': ['wheat'], 'detected_or_offi...",[],[]
4,4807770273704,Lucky Me,lm pc original 80g,"wheat flour, vegetable oil (palm oil, green te...","['wheat', 'soy']",[],[],[],"{'detected_only': ['soy', 'wheat'], 'detected_...",[],['coconut']


In [3]:
# Clean HTML tags from ingredients using utility function
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Convert detected allergens list to binary vector
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

print(f"Sample cleaned text: {df['ingredients_cleaned'].iloc[0][:200]}")
print(f"Label distribution (per class): {np.array(df['labels'].tolist()).sum(axis=0)}")

Sample cleaned text: water, lychee juice from concentrate 25%, coconut 25%, fructose, sucrose, acidifiers (citric acid, calcium lactate), lychee flavor, preservative e211, gelling agent: gellan gum, color: e129 - 0.0001% 
Label distribution (per class): [497  98  88  16 277 355  77  32]


## 3. Train/Validation/Test Split (Stratified)

Using utility function for multilabel stratified split

In [4]:
# Prepare data for splitting
X = df["ingredients_cleaned"].values
y = np.array(df["labels"].tolist())

# Create stratified splits using utility function
# Using 70% train, 15% validation, 15% test
train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=SEED
)

print(f"Train size: {len(train_texts)} (positive samples: {np.array(train_labels).sum(axis=0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {np.array(val_labels).sum(axis=0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {np.array(test_labels).sum(axis=0).sum()})")

Train size: 739 (positive samples: 1007)
Val size:   159   (positive samples: 216)
Test size:  159  (positive samples: 217)


## 4. Data Augmentation (Training Set Only)

Using utility function for data augmentation

In [5]:
# Create temporary dataframes for augmentation
train_df = pd.DataFrame({
    'text': train_texts,
    'labels': train_labels.tolist(),
    'ingredients_cleaned': train_texts  # Simplified for augmentation
})

# Apply data augmentation using utility function
train_aug_df = augment_dataframe(train_df, num_augmented=2)
print(f"Original training size: {len(train_df)}")
print(f"Augmented training size: {len(train_aug_df)}")

# Combine original training + augmented training
combined_train_df = pd.concat([train_df, train_aug_df], ignore_index=True)
print(f"Combined training size: {len(combined_train_df)}")

# Extract texts and labels for training
train_texts_final = combined_train_df["text"].tolist()
train_labels_final = np.array(combined_train_df["labels"].tolist())

# Validation and test sets remain unchanged
val_texts_final = val_texts
val_labels_final = np.array(val_labels)
test_texts_final = test_texts
test_labels_final = np.array(test_labels)

Original training size: 739
Augmented training size: 1213
Combined training size: 1952


## 5. Compute Class Weights for Imbalance

Using utility function to compute class weights

In [6]:
# Compute class weights using utility function
class_weights_tensor = compute_class_weights(train_labels_final)

print("Class weights:")
allergen_list = get_allergen_list()
for i, allergen in enumerate(allergen_list):
    print(f"  {allergen:10s}: {class_weights_tensor[i]:.3f}")

Class weights:
  milk      : 0.371
  eggs      : 1.027
  peanuts   : 1.072
  tree_nuts : 1.797
  soy       : 0.585
  wheat     : 0.501
  fish      : 1.152
  shellfish : 1.496


## 6. Tokenization

Determine optimal max_length based on 95th percentile of token lengths

In [7]:
model_name = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Compute max length from training texts
sample_lengths = [len(tokenizer.encode(t, truncation=False)) for t in train_texts_final[:500]]
max_len = int(np.percentile(sample_lengths, 95))
MAX_LEN = min(max_len, 256)
print(f"Using max_length = {MAX_LEN}")

def tokenize_data(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_encodings = tokenize_data(train_texts_final)
val_encodings = tokenize_data(val_texts_final)
test_encodings = tokenize_data(test_texts_final)

class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = FoodDataset(train_encodings, train_labels_final)
val_dataset = FoodDataset(val_encodings, val_labels_final)
test_dataset = FoodDataset(test_encodings, test_labels_final)

Using max_length = 221


## 7. Model Definition with Custom Trainer (Weighted BCE Loss)

Using a custom trainer to apply class weights

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(get_allergen_list()),
    problem_type="multi_label_classification"
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        # Weighted BCE loss
        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.class_weights.to(logits.device),
            reduction='mean'
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Optional: Focal Loss (uncomment to use instead)
# class FocalLossTrainer(Trainer):
#     def __init__(self, *args, alpha=0.25, gamma=2, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.alpha = alpha
#         self.gamma = gamma
#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs["labels"]
#         outputs = model(**inputs)
#         logits = outputs.logits
#         bce = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
#         pt = torch.exp(-bce)
#         focal_loss = self.alpha * (1-pt)**self.gamma * bce
#         loss = focal_loss.mean()
#         return (loss, outputs) if return_outputs else loss

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

[transformers] MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

## 8. Training Arguments with Warmup

We add `warmup_ratio=0.1` to stabilize early training

In [9]:
training_args = TrainingArguments(
    output_dir="./models/mobilebert_allergen",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to="tensorboard",
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = expit(logits)
    preds = (probs > 0.5).astype(int)
    return {
        "f1": f1_score(labels, preds, average="macro", zero_division=0)
    }

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    class_weights=class_weights_tensor
)

## 9. Train the Model

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,5.168793,6.470118,0.006494
2,0.150778,0.133736,0.647037
3,0.077583,0.085275,0.774602
4,0.050132,0.094263,0.780425
5,0.038544,0.093876,0.776647
6,0.031913,0.085921,0.791536
7,0.017019,0.101691,0.800748
8,0.019966,0.105292,0.788015
9,0.024068,0.095496,0.815006
10,0.016004,0.111235,0.811000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3660, training_loss=90743.27912752866, metrics={'train_runtime': 539.262, 'train_samples_per_second': 72.395, 'train_steps_per_second': 9.049, 'total_flos': 792657227082240.0, 'train_loss': 90743.27912752866, 'epoch': 15.0})

## 10. Evaluate on Validation Set (Baseline with 0.5 threshold)

In [12]:
val_output = trainer.predict(val_dataset)
val_logits = val_output.predictions
val_labels = val_output.label_ids

val_probs = expit(val_logits)
val_preds_05 = (val_probs >= 0.5).astype(int)

print("Baseline (threshold=0.5) Macro F1:", f1_score(val_labels, val_preds_05, average="macro", zero_division=0))

Baseline (threshold=0.5) Macro F1: 0.828432132788701


## 11. Per-Class Threshold Optimization

We find optimal thresholds on validation set to maximize macro F1 per class
Using utility function for threshold optimization

In [13]:
# Find optimal thresholds using utility function
best_thresholds = find_best_thresholds(val_probs, val_labels_final)
np.save("../models/best_thresholds.npy", best_thresholds)

print("Optimal thresholds:")
allergen_list = get_allergen_list()
for i, allergen in enumerate(allergen_list):
    print(f"{allergen:12s} → threshold={best_thresholds[i]:.2f}")

Optimal thresholds:
milk         → threshold=0.08
eggs         → threshold=0.57
peanuts      → threshold=0.04
tree_nuts    → threshold=0.01
soy          → threshold=0.17
wheat        → threshold=0.03
fish         → threshold=0.01
shellfish    → threshold=0.03


## 12. Apply Optimized Thresholds

Using utility function to apply thresholds

In [14]:
# Apply optimized thresholds using utility function
val_preds_opt = apply_thresholds(val_probs, best_thresholds)

print("\nValidation set after threshold tuning:")
print(f"Macro F1: {f1_score(val_labels_final, val_preds_opt, average='macro', zero_division=0):.4f}")
print(f"Micro F1: {f1_score(val_labels_final, val_preds_opt, average='micro', zero_division=0):.4f}")


Validation set after threshold tuning:
Macro F1: 0.9025
Micro F1: 0.9454


## 13. Final Evaluation on Test Set

Using utility functions for evaluation and error analysis

In [15]:
test_output = trainer.predict(test_dataset)
test_logits = test_output.predictions
test_labels_true = test_output.label_ids
test_probs = expit(test_logits)
test_preds = apply_thresholds(test_probs, best_thresholds)

print("\n=== Test Set Performance ===")
print_classification_report(test_labels_true, test_preds, get_allergen_list(), prefix="Test Set")

# Optional: Per-class metrics
print_per_class_metrics(test_labels_true, test_preds, get_allergen_list(), prefix="Per-class metrics on test set:")

# Optional: Error analysis
error_indices = error_analysis(test_texts_final, test_labels_true, test_preds, get_allergen_list(), max_errors=3)


=== Test Set Performance ===
=== Test Set ===
              precision    recall  f1-score   support

        milk     0.9595    0.9467    0.9530        75
        eggs     1.0000    0.9286    0.9630        14
     peanuts     0.8571    0.9231    0.8889        13
   tree_nuts     0.4000    0.6667    0.5000         3
         soy     0.8889    0.9524    0.9195        42
       wheat     0.9636    0.9815    0.9725        54
        fish     0.7333    1.0000    0.8462        11
   shellfish     0.7143    1.0000    0.8333         5

   micro avg     0.9079    0.9539    0.9303       217
   macro avg     0.8146    0.9249    0.8595       217
weighted avg     0.9185    0.9539    0.9337       217
 samples avg     0.6205    0.6345    0.6229       217

Macro F1: 0.8595
Micro F1: 0.9303


Per-class metrics on test set:
milk          Prec: 95.95%  Rec: 94.67%  F1: 95.30%
eggs          Prec: 100.00%  Rec: 92.86%  F1: 96.30%
peanuts       Prec: 85.71%  Rec: 92.31%  F1: 88.89%
tree_nuts     Prec: 40.0

## 14. Save Final Model

Saving model and tokenizer

In [16]:
final_dir = "../models/mobilebert_allergen_final"
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)
    
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model saved to {final_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ../models/mobilebert_allergen_final
